# 04 - Feature Engineering
**Air Quality Prediction and Health Risk Analysis — Feature Engineering**

This notebook transforms cleaned air-quality observations into machine-learning-ready features for the MSc dissertation workflow. Feature engineering is essential for time-series forecasting because pollution levels depend on recent history, seasonal cycles, and environmental conditions. The features created here are designed to improve the performance of downstream prediction models and to support the methodology chapter of the thesis.

## 1. Introduction
Feature engineering is the process of creating informative input variables from raw observations. For air-quality forecasting, this is especially important because the target variable is influenced by temporal patterns, lagged behaviour, and meteorological conditions.

In [7]:
from pathlib import Path
import sys
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn"])
    import seaborn as sns

sns.set_theme(style="whitegrid")
plt.style.use("seaborn-v0_8-darkgrid")

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "model_ready_air_quality.csv"

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output path:", OUTPUT_PATH)

Project root: C:\Users\phxac\Downloads\air-quality-forecasting
Processed dir: C:\Users\phxac\Downloads\air-quality-forecasting\data\processed
Output path: C:\Users\phxac\Downloads\air-quality-forecasting\data\processed\model_ready_air_quality.csv


## 2. Load Dataset
The notebook loads the processed dataset and checks its structure before engineering new features.

In [8]:
candidate_paths = [
    PROCESSED_DIR / "clean_air_quality.csv",
    PROCESSED_DIR / "processed_air_quality_data.csv",
    PROCESSED_DIR / "air_quality_data.csv",
    Path.cwd().resolve() / "data" / "processed" / "processed_air_quality_data.csv",
    Path.cwd().resolve() / "data" / "processed" / "engineered_air_quality_data.csv",
]

input_path = None
for path in candidate_paths:
    if path.exists():
        input_path = path
        break

if input_path is None:
    raise FileNotFoundError("No processed air-quality dataset found in the expected locations.")

print("Loading from:", input_path)
df = pd.read_csv(input_path)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("\nSample rows:")
display(df.head())

Loading from: C:\Users\phxac\Downloads\air-quality-forecasting\data\processed\processed_air_quality_data.csv
Dataset shape: (50, 10)
Columns: ['location', 'city', 'country', 'latitude', 'longitude', 'parameter', 'value', 'unit', 'date_utc', 'raw_measurement']

Sample rows:


,location,city,country,latitude,longitude,parameter,value,unit,date_utc,raw_measurement
0,unknown,unknown,unknown,38.767533,-119.798870,pm25,10.0,µg/m³,2017-08-20 18:00:00+00:00,"{""datetime"": {""utc"": ""2017-08-20T18:00:00Z"", ""..."
1,unknown,unknown,unknown,39.403903,-120.195751,pm25,1.0,µg/m³,2020-01-09 16:00:00+00:00,"{""datetime"": {""utc"": ""2020-01-09T16:00:00Z"", ""..."
2,unknown,unknown,unknown,36.247808,-121.781317,pm25,0.0,µg/m³,2020-11-24 16:00:00+00:00,"{""datetime"": {""utc"": ""2020-11-24T16:00:00Z"", ""..."
3,unknown,unknown,unknown,40.146500,117.070900,pm25,16.0,µg/m³,2021-08-09 11:00:00+00:00,"{""datetime"": {""utc"": ""2021-08-09T11:00:00Z"", ""..."
4,unknown,unknown,unknown,34.116400,108.611600,pm25,20.0,µg/m³,2021-08-09 11:00:00+00:00,"{""datetime"": {""utc"": ""2021-08-09T11:00:00Z"", ""..."


## 3. Temporal Feature Engineering
Air pollutants often vary by hour, day, month, and season. These time-based features help the model capture recurring cycles such as rush-hour traffic, heating demand, or seasonal atmospheric conditions.

In [9]:
datetime_column = None
for candidate in ["date_utc", "date", "timestamp", "datetime"]:
    if candidate in df.columns:
        datetime_column = candidate
        break

if datetime_column is None:
    raise ValueError("No datetime-like column was found in the dataset.")

df[datetime_column] = pd.to_datetime(df[datetime_column], errors="coerce", utc=True)
df = df.dropna(subset=[datetime_column]).sort_values(datetime_column).reset_index(drop=True)

df["hour"] = df[datetime_column].dt.hour
df["day"] = df[datetime_column].dt.day
df["month"] = df[datetime_column].dt.month
df["day_of_week"] = df[datetime_column].dt.dayofweek
df["weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    if month in [3, 4, 5]:
        return "spring"
    if month in [6, 7, 8]:
        return "summer"
    return "autumn"

df["season"] = df["month"].apply(get_season)
df["season_code"] = df["season"].map({"winter": 0, "spring": 1, "summer": 2, "autumn": 3})

print("Temporal features created.")
display(df[[datetime_column, "hour", "day", "month", "day_of_week", "weekend", "season"]].head())

Temporal features created.


,date_utc,hour,day,month,day_of_week,weekend,season
0,2017-08-20 18:00:00+00:00,18,20,8,6,1,summer
1,2020-01-09 16:00:00+00:00,16,9,1,3,0,winter
2,2020-11-24 16:00:00+00:00,16,24,11,1,0,autumn
3,2021-08-09 11:00:00+00:00,11,9,8,0,0,summer
4,2021-08-09 11:00:00+00:00,11,9,8,0,0,summer


## 4. Lag Features
Historical pollutant levels are often the strongest predictors of current and future concentrations. Lag features allow the model to learn persistence and short-term autocorrelation.

In [10]:
pollutant_candidates = ["pm25", "pm2_5", "pm2.5", "value", "concentration"]
selected_pollutant = None
for candidate in pollutant_candidates:
    if candidate in df.columns:
        selected_pollutant = candidate
        break

if selected_pollutant is None:
    raise KeyError("No suitable PM2.5-like target column was found.")

df[selected_pollutant] = pd.to_numeric(df[selected_pollutant], errors="coerce")
for lag in [1, 3, 6, 24]:
    df[f"{selected_pollutant}_lag_{lag}"] = df[selected_pollutant].shift(lag)

print(f"Lag features created for {selected_pollutant}.")
display(df[[selected_pollutant, f"{selected_pollutant}_lag_1", f"{selected_pollutant}_lag_3", f"{selected_pollutant}_lag_6", f"{selected_pollutant}_lag_24"]].head())

Lag features created for value.


,value,value_lag_1,value_lag_3,value_lag_6,value_lag_24
0,10.0,NaN,NaN,NaN,NaN
1,1.0,10.0,NaN,NaN,NaN
2,0.0,1.0,NaN,NaN,NaN
3,16.0,0.0,10.0,NaN,NaN
4,20.0,16.0,1.0,NaN,NaN


## 5. Rolling Statistics
Rolling averages and rolling variability help capture trailing pollution trends and short-term instability. These statistics are useful for identifying whether conditions are improving or deteriorating over time.

In [ ]:
n_rows = len(df)
window_24 = 24
window_7d = 24 * 7
min_24 = min(n_rows, window_24)
min_7d = min(n_rows, window_7d)

# Use adaptive rolling windows for small datasets.

df[f"{selected_pollutant}_rolling_mean_24h"] = (display(df[[selected_pollutant, f"{selected_pollutant}_rolling_mean_24h", f"{selected_pollutant}_rolling_std_24h", f"{selected_pollutant}_rolling_mean_7d", f"{selected_pollutant}_rolling_std_7d"]].head())

    df[selected_pollutant].rolling(window=window_24, min_periods=1).mean()    print("Rolling statistics added.")

)else:

df[f"{selected_pollutant}_rolling_std_24h"] = (    print(f"Dataset is small ({n_rows} rows). Using {min_7d}-row rolling windows for 7d features.")

    df[selected_pollutant].rolling(window=window_24, min_periods=1).std().fillna(0)if n_rows < window_7d:

)

df[f"{selected_pollutant}_rolling_mean_7d"] = ()

    df[selected_pollutant].rolling(window=min(window_7d, n_rows), min_periods=1).mean()    df[selected_pollutant].rolling(window=min(window_7d, n_rows), min_periods=1).std().fillna(0)

)df[f"{selected_pollutant}_rolling_std_7d"] = (

Rolling statistics added.


,value,value_rolling_mean_24h,value_rolling_std_24h,value_rolling_mean_7d,value_rolling_std_7d
0,10.0,NaN,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN,NaN
2,0.0,NaN,NaN,NaN,NaN
3,16.0,NaN,NaN,NaN,NaN
4,20.0,NaN,NaN,NaN,NaN


## 6. Environmental Feature Engineering
Meteorological variables and co-pollutants provide useful contextual information. Temperature, humidity, wind speed, pressure, and additional pollutant measures can help the model capture physical relationships between environmental conditions and PM2.5 concentration.

In [13]:
environmental_columns = ["temperature", "humidity", "wind_speed", "pressure", "wind", "temp", "rh"]
available_env = [col for col in environmental_columns if col in df.columns]

for col in available_env:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "temperature" in df.columns and "humidity" in df.columns:
    df["temperature_humidity_interaction"] = df["temperature"] * df["humidity"]

if "temperature" in df.columns and "pressure" in df.columns:
    df["temperature_pressure_interaction"] = df["temperature"] * df["pressure"]

for col in ["pm10", "no2", "so2", "co", "o3"]:
    if col in df.columns:
        df[f"{col}_lag_1"] = df[col].shift(1)

print("Environmental features created from:", available_env)
columns_to_show = [col for col in available_env if col in df.columns]
for col in ["temperature_humidity_interaction", "temperature_pressure_interaction"]:
    if col in df.columns:
        columns_to_show.append(col)
if columns_to_show:
    display(df[columns_to_show].head())
else:
    print("No environmental interaction features were created because the required columns were not available.")

Environmental features created from: []
No environmental interaction features were created because the required columns were not available.


## 7. Target Variable Creation
The forecasting target is defined as the next-step PM2.5 concentration. This avoids leakage because the target is based on a future observation relative to the input features used at the current timestep.

In [14]:
forecast_horizon = 1
df["target_pm25"] = df[selected_pollutant].shift(-forecast_horizon)
print("Target variable created as the next-step PM2.5 concentration.")
display(df[[selected_pollutant, "target_pm25"]].head())

Target variable created as the next-step PM2.5 concentration.


,value,target_pm25
0,10.0,1.0
1,1.0,0.0
2,0.0,16.0
3,16.0,20.0
4,20.0,23.3


## 8. Handle Missing Values
Lagged and rolling features create missing values at the start of the time series. These rows are removed to preserve data quality and maintain a chronologically consistent modelling dataset.

In [ ]:
feature_columns = [
    "hour", "day", "month", "day_of_week", "weekend", "season_code",
    f"{selected_pollutant}_lag_1", f"{selected_pollutant}_lag_3", f"{selected_pollutant}_lag_6", f"{selected_pollutant}_lag_24",
    f"{selected_pollutant}_rolling_mean_24h", f"{selected_pollutant}_rolling_std_24h",
    f"{selected_pollutant}_rolling_mean_7d", f"{selected_pollutant}_rolling_std_7d",
]
feature_columns += [col for col in ["temperature", "humidity", "wind_speed", "pressure", "wind", "temp", "rh"] if col in df.columns]
feature_columns += [col for col in ["temperature_humidity_interaction", "temperature_pressure_interaction"] if col in df.columns]
feature_columns += [col for col in df.columns if col.endswith("_lag_1") and col.startswith(("pm10", "no2", "so2", "co", "o3"))]

model_df = df.dropna(subset=["target_pm25"]).copy()
print("Rows after dropping rows with missing target values:", model_df.shape[0])

# Keep rows even if rolling statistics are partial, because adaptive rolling windows
# have already produced valid values for smaller datasets.
if model_df.empty:

    print("No rows remain after target filtering. Retaining all rows with a valid target where possible.")    model_df = df.copy()

Rows after dropping missing values: 0


## 9. Correlation Analysis
Correlation analysis helps identify which engineered features have the strongest relationship with the target variable. This supports feature selection and interpretation for the dissertation.

In [16]:
plot_columns = [col for col in feature_columns if col in model_df.columns]
plot_columns.append("target_pm25")

corr = model_df[plot_columns].corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

corr_with_target = corr["target_pm25"].drop("target_pm25").sort_values(ascending=False)
print("Top correlations with the target variable:")
print(corr_with_target.head(10))

c:\Users\phxac\Downloads\air-quality-forecasting\.venv\Lib\site-packages\seaborn\matrix.py:202: RuntimeWarning: All-NaN slice encountered
  vmin = np.nanmin(calc_data)
c:\Users\phxac\Downloads\air-quality-forecasting\.venv\Lib\site-packages\seaborn\matrix.py:207: RuntimeWarning: All-NaN slice encountered
  vmax = np.nanmax(calc_data)


Top correlations with the target variable:
hour           NaN
day            NaN
month          NaN
day_of_week    NaN
weekend        NaN
season_code    NaN
value_lag_1    NaN
value_lag_3    NaN
value_lag_6    NaN
value_lag_24   NaN
Name: target_pm25, dtype: float64


C:\Users\phxac\AppData\Local\Temp\ipykernel_28888\2659386402.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Save Dataset
The engineered dataset is now ready for model training. It contains temporal, lagged, rolling, and environmental variables together with the future-step target needed for supervised learning.

In [17]:
model_ready_df = model_df[[datetime_column, selected_pollutant, "target_pm25", *feature_columns]].copy()
model_ready_df.to_csv(OUTPUT_PATH, index=False)

print("Saved model-ready dataset to:", OUTPUT_PATH)
print("Final shape:", model_ready_df.shape)
print("Target variable:", "target_pm25")
model_ready_df.head()

Saved model-ready dataset to: C:\Users\phxac\Downloads\air-quality-forecasting\data\processed\model_ready_air_quality.csv
Final shape: (0, 17)
Target variable: target_pm25


,date_utc,value,target_pm25,hour,day,month,day_of_week,weekend,season_code,value_lag_1,value_lag_3,value_lag_6,value_lag_24,value_rolling_mean_24h,value_rolling_std_24h,value_rolling_mean_7d,value_rolling_std_7d
